# Clasificación de Tumores con Transfer Learning (ResNet18)
Este notebook muestra cómo adaptar un modelo preentrenado (ResNet18) para la clasificación de imágenes médicas usando fine-tuning.

Usaremos un dataset de ejemplo simplificado (puedes sustituirlo por Kather, MedMNIST o uno personalizado).

In [ ]:
# Instalación necesaria para Google Colab (si aplica)
# !pip install torch torchvision matplotlib scikit-learn

## 1. Cargar librerías y comprobar dispositivo

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt
import numpy as np
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

## 2. Preparar el dataset
Usaremos un conjunto de carpetas con imágenes clasificadas por subdirectorios (una carpeta por clase).

Organiza las imágenes como:
```
dataset/
    train/
        tumor/
        normal/
    val/
        tumor/
        normal/
```

In [ ]:
data_dir = "dataset"

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_data = datasets.ImageFolder(os.path.join(data_dir, 'train'), transform=transform)
val_data = datasets.ImageFolder(os.path.join(data_dir, 'val'), transform=transform)

train_loader = torch.utils.data.DataLoader(train_data, batch_size=16, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_data, batch_size=16, shuffle=False)

class_names = train_data.classes
print(f"Clases: {class_names}")

## 3. Cargar modelo ResNet18 y ajustar la última capa

In [ ]:
model = models.resnet18(pretrained=True)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, len(class_names))
model = model.to(device)

## 4. Entrenamiento del modelo

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0005)

for epoch in range(5):  # número de épocas reducido para pruebas rápidas
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Época {epoch+1}, pérdida: {running_loss/len(train_loader):.4f}")

## 5. Evaluación sobre conjunto de validación

In [ ]:
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for inputs, labels in val_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Precisión en validación: {100 * correct / total:.2f}%")